# Iteration 09 - Ring Fenced Acute Stroke Beds

## Aim

Compare the current acute-bed delay profile with a ring-fenced scenario in which acute stroke beds are reserved exclusively for stroke patients, for ward sizes between 10 and 15 beds.


## Prompt Used

In iteration 9, the final scenario to consider is the effect of ring fencing stroke beds on flow. In this scenario, scenarios stroke beds are reserved exclusively for patients suffering an acute stroke (so called ‘ring-fencing’). Consider scenarios where the ward has between 10 and 15 acute stroke beds, and compare the current situation with the situation where these beds are ring fenced. Present the results of the simulation in table form, similar to the table pasted from Monks et Al supplemenary results.


In [ ]:
from pathlib import Path
import sys

root = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

from stroke_sim.config import SimulationSettings
from stroke_sim.validation import (
    PUBLISHED_CURRENT_ADMISSIONS_ACUTE_10_TO_15,
    PUBLISHED_RING_FENCED_ACUTE,
    build_two_scenario_table,
    compare_to_published_table,
)
from stroke_sim.runner import run_replications


## Scenario Run

The ring-fenced scenario is interpreted as measuring delay against the audited stroke-only occupancy in the acute ward.


In [ ]:
settings = SimulationSettings(run_length_days=365 * 5, warm_up_days=365 * 3, replications=30)
audit = run_replications(settings=settings)


## Current Versus Ring Fenced Table


In [ ]:
current = compare_to_published_table(
    audit,
    column='acute_occupancy',
    published=PUBLISHED_CURRENT_ADMISSIONS_ACUTE_10_TO_15,
)
ring_fenced = compare_to_published_table(
    audit,
    column='acute_stroke_occupancy',
    published=PUBLISHED_RING_FENCED_ACUTE,
)
ring_fenced_table = build_two_scenario_table(
    current,
    ring_fenced,
    bed_label='No. acute beds',
    baseline_prefix='current',
    scenario_prefix='ring_fenced',
)
ring_fenced_table


## Short Summary


In [ ]:
summary = {
    'current_mean_abs_error': round(float(current['abs_error'].mean()), 4),
    'ring_fenced_mean_abs_error': round(float(ring_fenced['abs_error'].mean()), 4),
}
summary


## Tester Notes

- Ring fencing is implemented analytically by comparing delay against stroke-only acute occupancy rather than total acute occupancy.
- This is a simplified but defensible interpretation of the published scenario: non-stroke patients no longer compete for the reserved stroke beds.
- Results should therefore be discussed as a reconstruction of the scenario rather than an exact copy of the original software model.
- The displayed `p(delay)` values are rounded to 2 decimal places for presentation, but the `1 in every n` values are computed from the underlying unrounded probabilities.
- This means a row can legitimately show `p(delay) = 0.00` while still having a finite `1 in every n` value, which is consistent with how the published table behaves.
